In [19]:
import os
import time
import re 
from urllib.parse import urljoin, urlparse, parse_qs

import pandas as pd
import requests

from bs4 import BeautifulSoup as bs
from tqdm.notebook import tqdm

In [28]:
BASE = "https://gmnara.com"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
})

def get_soup(url, sleep_sec=0.3):
    """GET 요청 후 BeautifulSoup 반환 (간단 레이트리밋 포함)."""
    r = session.get(url, timeout=20)
    r.raise_for_status()
    time.sleep(sleep_sec)
    return bs(r.text, "html.parser")

In [29]:
# "업소정보" 팝업 처리
POPUP_RE = re.compile(r"openPopup\('([^']+)'\)")

def extract_popup_path_from_href(href: str):
    """
    1) javascript:openPopup('/?m=bbs&bid=cp&mod=view&uid=20259367&disp=popup');
    2) /?m=bbs&bid=cp&mod=view&uid=20259367&disp=popup
    모두 처리
    """
    if not href:
        return None

    m = POPUP_RE.search(href)
    if m:
        return m.group(1)

    # 직접 링크 케이스
    if "mod=view" in href and "uid=" in href:
        return href

    return None


def extract_uid_from_popup_url(popup_url: str):
    qs = parse_qs(urlparse(popup_url).query)
    uid = qs.get("uid", [None])[0]
    return uid

In [30]:
#################### 
# "업소정보" 링크 수집
####################
ADDRESS_HINT_RE = re.compile(
    r"(서울|경기|인천|부산|대전|대구|광주|울산|세종|강원|충북|충남|전북|전남|경북|경남|제주)\s+[^\n]{5,80}"
)

def parse_listing_page(listing_url: str):
    soup = get_soup(listing_url)

    a_tags = soup.find_all("a", string=lambda s: s and "업소정보" in s)

    # uid별로 한 행만 유지하되, 더 좋은 값이 나오면 채워 넣는 dict
    by_uid = {}  # external_id -> row dict

    for a in a_tags:
        href = a.get("href", "")
        popup_path = extract_popup_path_from_href(href)
        if not popup_path:
            continue

        detail_url = urljoin(BASE, popup_path)
        external_id = extract_uid_from_popup_url(detail_url)

        li = a.find_parent("li")

        # --- 업소명 후보 ---
        name_raw = None
        if li:
            title_el = li.select_one("div.title") or li.select_one(".title")
            if title_el:
                name_raw = title_el.get_text(" ", strip=True)

        # --- 주소 후보(card) ---
        address_raw_card = None
        if li:
            loc_el = li.select_one("div.location span") or li.select_one(".location span")
            if loc_el:
                address_raw_card = loc_el.get_text(" ", strip=True)
            else:
                li_text = li.get_text("\n", strip=True)
                for line in li_text.split("\n"):
                    if ADDRESS_HINT_RE.search(line):
                        address_raw_card = line.strip()
                        break

        # 처음 보는 uid: 생성
        if external_id not in by_uid:
            by_uid[external_id] = {
                "listing_url": listing_url,
                "detail_url": detail_url,
                "external_id": external_id,
                "name_raw": name_raw,
                "address_raw_card": address_raw_card,
            }
        else:
            # 있는 uid면 "더 좋은 값"을 merge (비어있을 때만 채우기)
            if (not by_uid[external_id].get("name_raw")) and name_raw:
                by_uid[external_id]["name_raw"] = name_raw
            if (not by_uid[external_id].get("address_raw_card")) and address_raw_card:
                by_uid[external_id]["address_raw_card"] = address_raw_card

            # detail_url이 더 정상적인 형태로 들어오면 업데이트(안전장치)
            if by_uid[external_id].get("detail_url") != detail_url:
                by_uid[external_id]["detail_url"] = detail_url

    return list(by_uid.values())


In [ ]:
# 공백(일반 공백/특수공백) 섞여도 잡히게 \s* 허용
LABEL_PATTERNS = {
    "category_raw": r"업\s*종\s*테마\s*:\s*(.+)",
    "name_detail": r"업\s*체\s*이름\s*:\s*(.+)",
    "phone": r"전\s*화\s*번호\s*:\s*(.+)",
    "hours": r"영\s*업\s*시간\s*:\s*(.+)",
    "tagline": r"한\s*줄\s*소\s*개\s*(?:\n|:)\s*(.+)",

    # 주소 후보 2개 case 
    "address_by_location": r"업\s*소\s*위\s*치\s*:\s*(.+)",
    "address_by_region": r"지\s*역\s*:\s*(.+)",
}


def parse_detail_popup(detail_url: str):
    soup = get_soup(detail_url)

    text = soup.get_text("\n", strip=True)
    text = re.sub(r"[ \t]+", " ", text)

    out = {}
    for k, pat in LABEL_PATTERNS.items():
        m = re.search(pat, text)
        out[k] = m.group(1).strip() if m else None

    out["address_raw"] = out.get("address_by_location") or out.get("address_by_region")

    # 지도 링크
    map_url_raw = None
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if any(x in href.lower() for x in ["map", "kakao", "naver", "google"]):
            map_url_raw = urljoin(BASE, href)
            break

    out["map_url_raw"] = map_url_raw
    return out


In [32]:
################
# 데이터프레임화
################
RAW_COLS = [
    "source_site", "crawled_at", "listing_url", "detail_url", "external_id",
    "name_raw", "category_raw", "address_raw", "map_url_raw", "status_raw"
]

def now_kst_str():
    # UTC+9 문자열(간단 버전)
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

def build_raw_row(listing_url, detail_url, external_id, name_raw_card=None, address_raw_card=None):
    detail = parse_detail_popup(detail_url)

    name_raw = detail.get("name_detail") or name_raw_card
    category_raw = detail.get("category_raw")
    address_raw = detail.get("address_raw") or address_raw_card

    row = {
        "source_site": "gmnara.com",
        "crawled_at": now_kst_str(),
        "listing_url": listing_url,
        "detail_url": detail_url,
        "external_id": external_id,
        "name_raw": name_raw,
        "category_raw": category_raw,
        "address_raw": address_raw,
        "map_url_raw": detail.get("map_url_raw"),
        "status_raw": None,
    }
    return row

In [33]:
###########################
# 실행
###########################

listing_urls = [
    "https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8",  # 강남
    "https://gmnara.com/b/cp?area=%EA%B0%95%EB%B6%81",   # 강북
    "https://gmnara.com/b/cp?area=%EA%B4%80%EC%95%85",  # 관악
    "https://gmnara.com/b/cp?area=%EA%B5%AC%EB%A1%9C",  # 구로
    "https://gmnara.com/b/cp?area=%EB%85%B8%EC%9B%90",  # 노원
    "https://gmnara.com/b/cp?area=%EB%8F%99%EB%8C%80%EB%AC%B8",     # 동대문
    "https://gmnara.com/b/cp?area=%EB%A7%88%ED%8F%AC",      # 마포
    "https://gmnara.com/b/cp?area=%EC%84%B1%EB%B6%81",      # 성북
    "https://gmnara.com/b/cp?area=%EC%96%91%EC%B2%9C",      # 양천
    "https://gmnara.com/b/cp?area=%EC%9D%80%ED%8F%89",  # 은평
    "https://gmnara.com/b/cp?area=%EC%A4%91%EA%B5%AC",  # 중구
    "https://gmnara.com/b/cp?area=%EA%B0%95%EB%8F%99",  # 강동
    "https://gmnara.com/b/cp?area=%EA%B0%95%EC%84%9C",  # 강서
    "https://gmnara.com/b/cp?area=%EA%B4%91%EC%A7%84",  # 광진
    "https://gmnara.com/b/cp?area=%EA%B8%88%EC%B2%9C",  # 금천
    "https://gmnara.com/b/cp?area=%EB%8F%84%EB%B4%89",  # 도봉
    "https://gmnara.com/b/cp?area=%EB%8F%99%EC%9E%91",  # 동작
    "https://gmnara.com/b/cp?area=%EC%84%9C%EC%B4%88",  # 서초
    "https://gmnara.com/b/cp?area=%EC%86%A1%ED%8C%8C",  # 송파
    "https://gmnara.com/b/cp?area=%EC%98%81%EB%93%B1%ED%8F%AC",  # 영등포
    "https://gmnara.com/b/cp?area=%EC%A2%85%EB%A1%9C",   # 종로
    "https://gmnara.com/b/cp?area=%EC%A4%91%EB%9E%91"   #중랑 
]

all_list_items = []
for u in listing_urls:
    all_list_items.extend(parse_listing_page(u))

print("list items:", len(all_list_items))
pd.DataFrame(all_list_items).head(3)

list items: 109


,listing_url,detail_url,external_id,name_raw,address_raw_card
0,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,20259367,[ 로드샵 ] 바나나테라피,서울 강남구 봉은사로 지하 501 (삼성동) 삼성중앙역 6번출구 도보 5분
1,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,10,[홈타이] 영앤핫 홈케어,"서울 강남구 강남대로 지하 396 (역삼동) 서울,경기,인천"
2,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,20259352,[로드샵] 구인마사지,서울 강북구 도봉로 지하 50 (미아동) 미아사거리역3번출구


In [34]:
rows = []
for item in tqdm(all_list_items):
    try:
        rows.append(
            build_raw_row(
                listing_url=item["listing_url"],
                detail_url=item["detail_url"],
                external_id=item["external_id"],
                name_raw_card=item.get("name_raw"),
                address_raw_card=item.get("address_raw_card"),
            )
        )
    except Exception as e:
        # 실패 로그를 남기고 계속 진행
        rows.append({
            "source_site": "gmnara.com",
            "crawled_at": now_kst_str(),
            "listing_url": item["listing_url"],
            "detail_url": item["detail_url"],
            "external_id": item["external_id"],
            "name_raw": item.get("name_raw"),
            "category_raw": None,
            "address_raw": item.get("address_raw_card"),
            "map_url_raw": None,
            "status_raw": f"ERROR: {type(e).__name__}",
        })

df_raw = pd.DataFrame(rows, columns=RAW_COLS)
df_raw.head(5)

  0%|          | 0/109 [00:00<?, ?it/s]

,source_site,crawled_at,listing_url,detail_url,external_id,name_raw,category_raw,address_raw,map_url_raw,status_raw
0,gmnara.com,2026-01-19 17:35:34,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,20259367,바나나테라피,로드샵,서울 강남구 봉은사로 지하 501 (삼성동) 삼성중앙역 6번출구 도보 5분,https://www.google.co.kr/search?q=건마나라,None
1,gmnara.com,2026-01-19 17:35:35,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,10,영앤핫 홈케어,홈타이,"서울 강남구 강남대로 지하 396 (역삼동) 서울,경기,인천",https://www.google.co.kr/search?q=건마나라,None
2,gmnara.com,2026-01-19 17:35:35,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,20259352,구인마사지,로드샵,서울 강북구 도봉로 지하 50 (미아동) 미아사거리역3번출구,https://www.google.co.kr/search?q=건마나라,None
3,gmnara.com,2026-01-19 17:35:36,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,20259584,5월스파,로드샵,서울 강남구 학동로33길 15 (논현동),https://www.google.co.kr/search?q=건마나라,None
4,gmnara.com,2026-01-19 17:35:37,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,20259372,더원왁싱,왁싱,서울 강남구 강남대로 328 (역삼동) 강남역3번출구,https://www.google.co.kr/search?q=건마나라,None


In [35]:
##############################
# CSV 저장
##############################
from datetime import datetime

out_dir = "../data"
os.makedirs(out_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = os.path.join(out_dir, f"gmnara_raw_{timestamp}.csv")

df_raw.to_csv(out_path, index=False, encoding="utf-8-sig")

print("saved:", os.path.abspath(out_path))

saved: c:\DSJA\2025-sbsDSJA\TeamProject\data\gmnara_raw_20260119_173732.csv
